In [229]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, balanced_accuracy_score

In [230]:
data = pd.read_csv("Data/USO/USO_clean.csv")

**Statistics**

Time gap from last candle

In [231]:
data["timestamp"] = pd.to_datetime(data["timestamp"])
data["time_gap"] = data["timestamp"].diff().dt.total_seconds()/3600

Change hour to hour

In [232]:
data["internal_return"] = (data["close"] - data["open"])/ data["open"]

data["cc_return"] = data["close"].pct_change()

Volume

In [233]:
data["hour"] = pd.to_datetime(data["timestamp"]).dt.hour

hourly_typical_volume = (
    data.groupby("hour")["volume"]
      .transform(lambda s: s.shift(1).rolling(10, min_periods=5).mean())
)

data["hourly_relative_volume"] = (data["volume"]/hourly_typical_volume.replace(0, np.nan))

In [234]:
data["relative_volume"] = data["volume"]/data["volume"].shift(1).rolling(10, min_periods=5).mean()

Volatility

In [235]:
window = 10
min_periods = 5
data["volatility"] = data["cc_return"].rolling(window, min_periods=min_periods).std()

Target

In [236]:
horizon = 1

future_close = data["close"].shift(-horizon)

data[f"target_{horizon}"] = np.where(
    future_close.notna(),
    (future_close > data["close"]).astype(int),
    np.nan
)

Correlations

In [237]:
X = data.drop(columns = f"target_{horizon}")
split_id = int(len(data) * 0.8)
X_train = X.iloc[:split_id]
y_train = data[f"target_{horizon}"].iloc[:split_id]

In [238]:
feature_target_corr = (
    X_train.assign(target_1=y_train)
    .corr(numeric_only=True)[f"target_{horizon}"]
    .drop(f"target_{horizon}")
    .sort_values(key=abs, ascending=False)
)

print(feature_target_corr)

cc_return                -0.037278
internal_return          -0.034487
time_gap                 -0.009933
relative_volume           0.009175
open                      0.009063
high                      0.009022
low                       0.008925
close                     0.008837
hour                     -0.007468
volatility                0.006621
volume                    0.004140
hourly_relative_volume   -0.002723
Name: target_1, dtype: float64


data = data.sort_index()

data["previous_direction"] = (
    data["close"].diff().shift(1) > 0
).astype(int)

valid = data[f"target_{horizon}"].notna()

y_true = data.loc[valid, f"target_{horizon}"].astype(int)
y_prev = data.loc[valid, "previous_direction"]

print("Previous-direction accuracy:", accuracy_score(y_true, y_prev))
print("Previous-direction balanced accuracy:",
      balanced_accuracy_score(y_true, y_prev))

Target distribution

In [239]:
print("Target distribution:")
print(y_train.value_counts(normalize=True))

print("\nFeature summary:")
print(X_train.describe().T)

print("\nMissing values:")
print(X_train.isna().sum().sort_values(ascending=False).head(10))

Target distribution:
target_1
0.0    0.518526
1.0    0.481474
Name: proportion, dtype: float64

Feature summary:
                          count           mean            std        min  \
open                    57783.0     207.113696     167.648876  15.760000   
high                    57783.0     207.784330     168.245571  16.880000   
low                     57783.0     206.419361     167.031280  15.760000   
close                   57783.0     207.106654     167.642912  16.320000   
volume                  57783.0  202174.398992  471503.111727  12.500000   
time_gap                57782.0       2.544841       7.891069   1.000000   
internal_return         57783.0      -0.000019       0.005775  -0.119966   
cc_return               57782.0      -0.000014       0.006542  -0.217241   
hour                    57783.0      15.652597       4.637349   0.000000   
hourly_relative_volume  57696.0       1.288151       4.131473   0.000099   
relative_volume         57778.0       1.706757     

Extreme values

In [240]:
data = data.sort_values("timestamp").copy()

data["cc_return"] = data["close"].pct_change()

print(
    data.loc[
        data["cc_return"].abs() > 0.20,
        [
            "timestamp", "open", "high", "low", "close",
            "volume", "cc_return"
        ]
    ].to_string(index=False)
)

                timestamp  open  high  low  close   volume  cc_return
2020-03-09 08:00:00+00:00  55.2  55.2 50.4  54.48 89155.75  -0.217241


In [241]:
data = data.sort_values("timestamp").reset_index(drop=True)

for ts in [
    "2020-03-13 19:00:00+00:00",
    "2025-04-09 17:00:00+00:00"
]:
    ts = pd.Timestamp(ts)
    idx = data.index[data["timestamp"] == ts]

    if len(idx):
        i = idx[0]
        print(f"\n--- Around {ts} ---")
        print(
            data.iloc[max(0, i - 3):i + 4][
                ["timestamp", "open", "high", "low", "close",
                 "volume", "cc_return", f"target_{horizon}"]
            ].to_string(index=False)
        )


--- Around 2020-03-13 19:00:00+00:00 ---
                timestamp  open  high   low  close     volume  cc_return  target_1
2020-03-13 16:00:00+00:00 52.96 54.08 52.32  53.36  799830.38   0.009077       1.0
2020-03-13 17:00:00+00:00 53.36 54.40 53.12  54.32  713736.62   0.017991       0.0
2020-03-13 18:00:00+00:00 54.40 54.40 53.04  53.64  606429.88  -0.012518       1.0
2020-03-13 19:00:00+00:00 53.62 57.04 53.12  55.68 3659642.38   0.038031       0.0
2020-03-13 20:00:00+00:00 55.60 56.00 54.96  55.44  145138.12  -0.004310       1.0
2020-03-13 21:00:00+00:00 55.28 56.24 54.48  55.92   49915.12   0.008658       1.0
2020-03-13 22:00:00+00:00 55.92 56.40 55.60  56.24   26769.88   0.005722       1.0

--- Around 2025-04-09 17:00:00+00:00 ---
                timestamp  open  high   low  close    volume  cc_return  target_1
2025-04-09 14:00:00+00:00 62.11 62.35 60.67  60.72 1978218.0  -0.021907       1.0
2025-04-09 15:00:00+00:00 60.70 62.22 60.68  61.87 1090896.0   0.018939       1.0
2025-0

In [242]:
data = data.drop(columns = ["timestamp", "open", "high", "low", "close", "volume"])
data.to_csv("Data/USO/USO_stats.csv")